# VinDr-Mammo NSGA-III Optimization
## Multi-Objective Hyperparameter Optimization for Breast Cancer Detection

**Purpose:** Run multi-objective NSGA-III optimization on VinDr-Mammo dataset

**Objectives:**
1. Maximize PR-AUC
2. Maximize AUROC  
3. Minimize Brier Score
4. Minimize Robustness Degradation

**Two Optimization Modes:**
- **Standard NSGA-III**: Full evaluations for all candidates (expensive, accurate)
- **Surrogate-Assisted**: GP surrogates to reduce evaluations by ~70% (efficient, approximates well)

**Standard NSGA-III Configurations:**
- Demo: 5 pop × 3 gen = 15 evaluations (~4 hours)
- Recommended: 8 pop × 10 gen = 80 evaluations (~17 days)
- Full: 20 pop × 50 gen = 1,000 evaluations (~292 days)

**Surrogate-Assisted Configurations:**
- Recommended: 20 pop × 50 gen = 295 evaluations (~73 days, 70% faster)
- Demo: 10 pop × 10 gen = 70 evaluations (~15 days)

## 1. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies with compatible versions
# IMPORTANT: Install SymPy BEFORE PyTorch to avoid compatibility issues
!pip install -q 'sympy>=1.12'
!pip install -q pydicom opencv-python-headless scikit-image
!pip install -q pymoo
!pip install -q scikit-learn scipy
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

print("✓ Dependencies installed")
print("\nIMPORTANT: If you see 'AttributeError: module sympy has no attribute printing',")
print("           restart runtime: Runtime > Restart runtime, then re-run all cells")

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from datetime import datetime

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive & Setup Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Project path
PROJECT_PATH = "/content/drive/MyDrive/breast_cancer_detection"

if os.path.exists(PROJECT_PATH):
    print(f"✓ Project found: {PROJECT_PATH}")
    sys.path.insert(0, PROJECT_PATH)
else:
    raise FileNotFoundError(f"Project not found at {PROJECT_PATH}")

# Data paths
VINDR_IMAGES_ROOT = "/content/drive/MyDrive/vindr-mammo/images"
VINDR_CSV = "/content/drive/MyDrive/vindr-mammo/metadata/stratified_selection.csv"

# Verify
assert os.path.exists(VINDR_IMAGES_ROOT), f"VinDr images not found: {VINDR_IMAGES_ROOT}"
assert os.path.exists(VINDR_CSV), f"VinDr CSV not found: {VINDR_CSV}"

print("✓ All paths verified")

## 3. Configuration

**Choose your optimization approach:**

### Option A: Standard NSGA-III (Expensive, Baseline)
- **DEMO**: 5×3 = 15 evals, ~4 hours
- **RECOMMENDED**: 8×10 = 80 evals, ~17 days
- **FULL**: 20×50 = 1,000 evals, ~292 days

### Option B: Surrogate-Assisted NSGA-III (70% Faster)
- **DEMO**: 10×10, 70 true evals, ~15 days
- **RECOMMENDED**: 20×50, 295 true evals, ~73 days (70% savings)

**Surrogate strategy:**
- Phase 1: 3 initial generations with full evaluations (60 samples)
- Phase 2: GP surrogate predictions + 5 best candidates/generation for true eval

In [ ]:
# ============================================================================
# OPTIMIZATION CONFIGURATION
# ============================================================================

# Choose optimization mode: "STANDARD" or "SURROGATE"
OPTIMIZATION_MODE = "SURROGATE"  # Change this

# Choose scale: "DEMO", "RECOMMENDED", or "FULL"
OPTIMIZATION_SCALE = "RECOMMENDED"  # Change this

print("="*80)
print(f"OPTIMIZATION MODE: {OPTIMIZATION_MODE}")
print(f"OPTIMIZATION SCALE: {OPTIMIZATION_SCALE}")
print("="*80)

if OPTIMIZATION_MODE == "STANDARD":
    # Standard NSGA-III configurations
    if OPTIMIZATION_SCALE == "DEMO":
        CONFIG = {
            'population': 5,
            'generations': 3,
            'max_epochs': 5,
            'patience': 3,
            'batch_size': 4,
            'num_workers': 2,
            'use_full_dataset': False,
            'run_id': f"standard_demo_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        }
        total_evals = 15
        est_time = "4 hours"
        
    elif OPTIMIZATION_SCALE == "RECOMMENDED":
        CONFIG = {
            'population': 8,
            'generations': 10,
            'max_epochs': 50,
            'patience': 10,
            'batch_size': 8,
            'num_workers': 2,
            'use_full_dataset': True,
            'run_id': f"standard_recommended_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        }
        total_evals = 80
        est_time = "17 days"
        
    elif OPTIMIZATION_SCALE == "FULL":
        CONFIG = {
            'population': 20,
            'generations': 50,
            'max_epochs': 50,
            'patience': 10,
            'batch_size': 8,
            'num_workers': 2,
            'use_full_dataset': True,
            'run_id': f"standard_full_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        }
        total_evals = 1000
        est_time = "292 days"
    else:
        raise ValueError(f"Invalid scale: {OPTIMIZATION_SCALE}")
        
elif OPTIMIZATION_MODE == "SURROGATE":
    # Surrogate-assisted configurations
    if OPTIMIZATION_SCALE == "DEMO":
        CONFIG = {
            'population': 10,
            'generations': 10,
            'n_gen_init': 2,          # Initial sampling generations
            'n_true_per_gen': 5,      # True evaluations per gen (surrogate phase)
            'acquisition': 'uncertainty',  # Acquisition function
            'max_epochs': 50,
            'patience': 10,
            'batch_size': 8,
            'num_workers': 2,
            'use_full_dataset': True,
            'run_id': f"surrogate_demo_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        }
        total_evals = 10*2 + 8*5  # Phase 1 + Phase 2
        est_time = "15 days"
        
    elif OPTIMIZATION_SCALE == "RECOMMENDED":
        CONFIG = {
            'population': 20,
            'generations': 50,
            'n_gen_init': 3,          # Initial sampling generations
            'n_true_per_gen': 5,      # True evaluations per gen (surrogate phase)
            'acquisition': 'uncertainty',  # Acquisition function
            'max_epochs': 50,
            'patience': 10,
            'batch_size': 8,
            'num_workers': 2,
            'use_full_dataset': True,
            'run_id': f"surrogate_recommended_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        }
        total_evals = 20*3 + 47*5  # Phase 1 (60) + Phase 2 (235)
        est_time = "73 days (70% savings)"
    else:
        raise ValueError(f"Invalid scale for surrogate mode: {OPTIMIZATION_SCALE}")
else:
    raise ValueError(f"Invalid mode: {OPTIMIZATION_MODE}")

print(f"\nConfiguration:")
print(f"  Mode: {OPTIMIZATION_MODE}")
print(f"  Population: {CONFIG['population']}")
print(f"  Generations: {CONFIG['generations']}")
if OPTIMIZATION_MODE == "SURROGATE":
    print(f"  Initial sampling gens: {CONFIG['n_gen_init']}")
    print(f"  True evals/gen (surrogate phase): {CONFIG['n_true_per_gen']}")
    print(f"  Acquisition: {CONFIG['acquisition']}")
print(f"  Total true evaluations: {total_evals}")
print(f"  Max epochs per model: {CONFIG['max_epochs']}")
print(f"  Patience: {CONFIG['patience']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Full dataset: {CONFIG['use_full_dataset']}")
print(f"  Run ID: {CONFIG['run_id']}")
print(f"\nEstimated time: {est_time}")
print("="*80)

## 4. Load Dataset

In [ ]:
from src.preprocessing import MammographyPreprocessor
from src.datasets import VinDRMammoBinaryDataset, create_breast_level_splits

print("Loading VinDr-Mammo dataset...\n")

# Create preprocessor
preprocessor = MammographyPreprocessor(
    target_size=(720, 480),
    aspect_ratio=1.5
)

# Load full dataset
dataset = VinDRMammoBinaryDataset(
    images_root=VINDR_IMAGES_ROOT,
    csv_file=VINDR_CSV,
    preprocessor=preprocessor
)

print(f"Total samples: {len(dataset)}")

# Get labels for class imbalance
all_labels = [dataset[i][1].item() for i in range(len(dataset))]
n_benign = sum([1 for l in all_labels if l == 0])
n_malignant = sum([1 for l in all_labels if l == 1])

print(f"  Benign: {n_benign}")
print(f"  Malignant: {n_malignant}")
print(f"  Ratio: {n_benign/n_malignant:.2f}:1")

# Create breast-level split
print("\nCreating breast-level train/val split...")
train_dataset, val_dataset = create_breast_level_splits(
    dataset=dataset,
    train_ratio=0.8,
    random_state=42,
    stratify=True
)

print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print("\n✓ Dataset loaded successfully")

## 5. Run Optimization

**This cell dispatches to the appropriate optimization method based on your configuration.**

**Note:** For surrogate-assisted mode, progress is tracked and checkpoints are saved every 5 generations.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# Setup output directory
output_dir = os.path.join(PROJECT_PATH, "results", "nsga3_optimization")
os.makedirs(output_dir, exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"Run ID: {CONFIG['run_id']}\n")

if OPTIMIZATION_MODE == "STANDARD":
    # ========================================================================
    # STANDARD NSGA-III OPTIMIZATION
    # ========================================================================
    
    from src.optimization import BreastCancerOptimizationProblem
    from pymoo.algorithms.moo.nsga3 import NSGA3
    from pymoo.optimize import minimize
    from pymoo.util.ref_dirs import get_reference_directions
    import pickle
    
    print("="*80)
    print("STANDARD NSGA-III OPTIMIZATION")
    print("="*80)
    
    # Create optimization problem
    problem = BreastCancerOptimizationProblem(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        device=device,
        batch_size=CONFIG['batch_size'],
        num_workers=CONFIG['num_workers'],
        patience=CONFIG['patience'],
        max_epochs=CONFIG['max_epochs'],
        pos_weight=n_benign/n_malignant,
        random_seed=42
    )
    
    print(f"\n✓ Problem created")
    print(f"  Decision variables: {problem.n_var}")
    print(f"  Objectives: {problem.n_obj}")
    
    # Generate reference directions
    ref_dirs = get_reference_directions("das-dennis", 4, n_partitions=3)
    print(f"  Reference directions: {len(ref_dirs)}")
    
    # Create algorithm
    algorithm = NSGA3(
        ref_dirs=ref_dirs,
        pop_size=CONFIG['population']
    )
    
    print(f"\nRunning optimization...")
    print(f"  Generations: {CONFIG['generations']}")
    print(f"  Population: {CONFIG['population']}")
    print(f"  Expected evaluations: {CONFIG['population'] * CONFIG['generations']}\n")
    
    # Run optimization
    res = minimize(
        problem,
        algorithm,
        ('n_gen', CONFIG['generations']),
        verbose=True,
        save_history=True,
        seed=42
    )
    
    # Save results
    results_path = os.path.join(output_dir, f"results_{CONFIG['run_id']}.pkl")
    with open(results_path, 'wb') as f:
        pickle.dump(res, f)
    
    print("\n" + "="*80)
    print("OPTIMIZATION COMPLETE")
    print("="*80)
    print(f"\nTotal evaluations: {res.algorithm.n_eval}")
    print(f"Pareto solutions: {len(res.F)}")
    print(f"\nResults saved to: {results_path}")
    
elif OPTIMIZATION_MODE == "SURROGATE":
    # ========================================================================
    # SURROGATE-ASSISTED NSGA-III OPTIMIZATION
    # ========================================================================
    
    from src.optimization import BreastCancerOptimizationProblem
    from src.surrogate_optimizer import MultiObjectiveGPSurrogate
    from src.acquisition import get_acquisition_function
    from pymoo.algorithms.moo.nsga3 import NSGA3
    from pymoo.optimize import minimize
    from pymoo.util.ref_dirs import get_reference_directions
    import pickle
    
    print("="*80)
    print("SURROGATE-ASSISTED NSGA-III OPTIMIZATION")
    print("="*80)
    
    # Create optimization problem
    problem = BreastCancerOptimizationProblem(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        device=device,
        batch_size=CONFIG['batch_size'],
        num_workers=CONFIG['num_workers'],
        patience=CONFIG['patience'],
        max_epochs=CONFIG['max_epochs'],
        pos_weight=n_benign/n_malignant,
        random_seed=42,
        use_surrogate=False,
        surrogate_manager=None
    )
    
    print(f"\n✓ Problem created")
    print(f"  Decision variables: {problem.n_var}")
    print(f"  Objectives: {problem.n_obj}")
    
    # Generate reference directions
    ref_dirs = get_reference_directions("das-dennis", 4, n_partitions=3)
    print(f"  Reference directions: {len(ref_dirs)}")
    
    # Initialize algorithm and surrogate
    algorithm = NSGA3(ref_dirs=ref_dirs, pop_size=CONFIG['population'])
    surrogate = MultiObjectiveGPSurrogate(n_objectives=4, n_vars=5)
    acquisition_fn = get_acquisition_function(CONFIG['acquisition'])
    
    # Calculate expected budget
    phase1_evals = CONFIG['population'] * CONFIG['n_gen_init']
    phase2_gens = CONFIG['generations'] - CONFIG['n_gen_init']
    phase2_evals = phase2_gens * CONFIG['n_true_per_gen']
    total_expected = phase1_evals + phase2_evals
    baseline_evals = CONFIG['population'] * CONFIG['generations']
    savings = baseline_evals - total_expected
    
    print(f"\nExpected budget:")
    print(f"  Phase 1: {phase1_evals} evaluations ({CONFIG['n_gen_init']} gen)")
    print(f"  Phase 2: {phase2_evals} evaluations ({phase2_gens} gen × {CONFIG['n_true_per_gen']})")
    print(f"  Total: {total_expected} evaluations")
    print(f"  Baseline: {baseline_evals} evaluations")
    print(f"  Savings: {savings} evaluations ({100*savings/baseline_evals:.1f}%)")
    
    # Phase 1: Initial sampling
    print("\n" + "="*80)
    print("PHASE 1: INITIAL SAMPLING")
    print("="*80)
    
    problem.use_surrogate = False
    
    print(f"\nRunning {CONFIG['n_gen_init']} generations with true evaluations...\n")
    
    res_init = minimize(
        problem,
        algorithm,
        termination=("n_gen", CONFIG['n_gen_init']),
        seed=42,
        verbose=True,
        save_history=True
    )
    
    # Update surrogate
    X_init, F_init = problem.get_evaluation_history()
    surrogate.update(X_init, F_init)
    surrogate.fit()
    
    print(f"\n✓ Phase 1 complete")
    print(f"  True evaluations: {len(X_init)}")
    print(f"  GP surrogates fitted")
    
    # Phase 2: Surrogate-assisted
    print("\n" + "="*80)
    print("PHASE 2: SURROGATE-ASSISTED OPTIMIZATION")
    print("="*80)
    
    current_gen = CONFIG['n_gen_init']
    
    while current_gen < CONFIG['generations']:
        current_gen += 1
        print(f"\n{'='*80}")
        print(f"Generation {current_gen}/{CONFIG['generations']}")
        print(f"{'='*80}")
        
        # Generate offspring
        offspring = algorithm.infill()
        X_offspring = offspring.get("X")
        
        print(f"\n[1/5] Generated {len(X_offspring)} offspring candidates")
        
        # Predict with surrogate
        print(f"[2/5] Predicting objectives with GP surrogates...")
        F_mean, F_std = surrogate.predict(X_offspring, return_std=True)
        
        # Select candidates
        print(f"[3/5] Selecting top {CONFIG['n_true_per_gen']} candidates via '{CONFIG['acquisition']}'...")
        
        if CONFIG['acquisition'] == 'ei':
            F_current_best = F_init.min(axis=0)
            selected_idx = acquisition_fn(F_mean, F_std, CONFIG['n_true_per_gen'], F_current_best)
        else:
            selected_idx = acquisition_fn(F_mean, F_std, CONFIG['n_true_per_gen'])
        
        # True evaluation
        print(f"[4/5] Evaluating {len(selected_idx)} candidates with true function...")
        problem.use_surrogate = False
        X_selected = X_offspring[selected_idx]
        F_selected = []
        
        for i, x in enumerate(X_selected):
            print(f"      [{i+1}/{len(X_selected)}] Training model...")
            out_dict = {}
            problem._evaluate(np.array([x]), out_dict)
            F_selected.append(out_dict["F"][0])
        
        F_selected = np.array(F_selected)
        
        # Update surrogate
        surrogate.update(X_selected, F_selected)
        surrogate.fit()
        
        print(f"[5/5] Surrogate updated")
        print(f"      Training size: {surrogate.get_training_size()}")
        
        # Evaluate all offspring
        problem.use_surrogate = True
        problem.surrogate_manager = surrogate
        
        F_offspring = []
        for i, x in enumerate(X_offspring):
            if i in selected_idx:
                idx_in_selected = np.where(selected_idx == i)[0][0]
                F_offspring.append(F_selected[idx_in_selected])
            else:
                out_dict = {}
                problem._evaluate(np.array([x]), out_dict)
                F_offspring.append(out_dict["F"][0])
        
        offspring.set("F", np.array(F_offspring))
        
        # Advance algorithm
        algorithm.advance(infills=offspring)
        
        # Statistics
        total_true_evals = len(problem.eval_history_X)
        budget_used = total_true_evals / baseline_evals * 100
        
        print(f"\nGeneration {current_gen} complete:")
        print(f"  Total true evaluations: {total_true_evals}/{baseline_evals} ({budget_used:.1f}%)")
        print(f"  Pareto front size: {len(algorithm.pop.get('F'))}")
        
        # Checkpoint every 5 generations
        if current_gen % 5 == 0:
            checkpoint_path = os.path.join(output_dir, f"checkpoint_{CONFIG['run_id']}_gen{current_gen}.pkl")
            with open(checkpoint_path, 'wb') as f:
                pickle.dump({
                    'algorithm': algorithm,
                    'surrogate': surrogate,
                    'problem_history': problem.get_evaluation_history(),
                    'generation': current_gen,
                    'n_true_evals': total_true_evals
                }, f)
            print(f"  Checkpoint saved: checkpoint_gen{current_gen}.pkl")
    
    # Final results
    X_all, F_all = problem.get_evaluation_history()
    
    print("\n" + "="*80)
    print("OPTIMIZATION COMPLETE")
    print("="*80)
    
    print(f"\nFinal Statistics:")
    print(f"  Total true evaluations: {len(X_all)}/{baseline_evals}")
    print(f"  Budget saved: {baseline_evals - len(X_all)} evaluations ({100*(baseline_evals-len(X_all))/baseline_evals:.1f}%)")
    print(f"  Pareto solutions: {len(algorithm.pop.get('F'))}")
    
    # Save final results
    results_path = os.path.join(output_dir, f"results_{CONFIG['run_id']}.pkl")
    final_results = {
        'X_history': X_all,
        'F_history': F_all,
        'algorithm': algorithm,
        'surrogate': surrogate,
        'pop_X': algorithm.pop.get("X"),
        'pop_F': algorithm.pop.get("F"),
        'n_true_evals': len(X_all),
        'n_gen': CONFIG['generations'],
        'config': CONFIG
    }
    
    with open(results_path, 'wb') as f:
        pickle.dump(final_results, f)
    
    print(f"\n✓ Results saved to: {results_path}")
    
    # Create res object for compatibility with visualization cells
    class SurrogateResult:
        def __init__(self, algorithm):
            self.algorithm = algorithm
            self.X = algorithm.pop.get("X")
            self.F = algorithm.pop.get("F")
    
    res = SurrogateResult(algorithm)

print("\n✓ Optimization complete!")

## 6. Visualize Pareto Front

In [ ]:
import matplotlib.pyplot as plt

if res is not None:
    print("="*80)
    print("PARETO FRONT SOLUTIONS")
    print("="*80)
    
    # Display table
    print(f"\n{'ID':<5} {'PR-AUC':>8} {'AUROC':>8} {'Brier':>8} {'Robust':>8}")
    print("-" * 45)
    
    for i, f in enumerate(res.F):
        pr_auc = -f[0]
        auroc = -f[1]
        brier = f[2]
        robust = f[3]
        
        print(f"{i:<5} {pr_auc:>8.4f} {auroc:>8.4f} {brier:>8.4f} {robust:>8.4f}")
    
    # Visualize
    pr_auc = -res.F[:, 0]
    auroc = -res.F[:, 1]
    brier = res.F[:, 2]
    robust = res.F[:, 3]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    axes[0, 0].scatter(pr_auc, auroc, c='blue', s=100, alpha=0.6)
    axes[0, 0].set_xlabel('PR-AUC')
    axes[0, 0].set_ylabel('AUROC')
    axes[0, 0].set_title('PR-AUC vs AUROC')
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].scatter(pr_auc, brier, c='red', s=100, alpha=0.6)
    axes[0, 1].set_xlabel('PR-AUC')
    axes[0, 1].set_ylabel('Brier Score')
    axes[0, 1].set_title('PR-AUC vs Brier')
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[0, 2].scatter(pr_auc, robust, c='green', s=100, alpha=0.6)
    axes[0, 2].set_xlabel('PR-AUC')
    axes[0, 2].set_ylabel('Robustness Degradation')
    axes[0, 2].set_title('PR-AUC vs Robustness')
    axes[0, 2].grid(True, alpha=0.3)
    
    axes[1, 0].scatter(auroc, brier, c='purple', s=100, alpha=0.6)
    axes[1, 0].set_xlabel('AUROC')
    axes[1, 0].set_ylabel('Brier Score')
    axes[1, 0].set_title('AUROC vs Brier')
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].scatter(auroc, robust, c='orange', s=100, alpha=0.6)
    axes[1, 1].set_xlabel('AUROC')
    axes[1, 1].set_ylabel('Robustness Degradation')
    axes[1, 1].set_title('AUROC vs Robustness')
    axes[1, 1].grid(True, alpha=0.3)
    
    axes[1, 2].scatter(brier, robust, c='brown', s=100, alpha=0.6)
    axes[1, 2].set_xlabel('Brier Score')
    axes[1, 2].set_ylabel('Robustness Degradation')
    axes[1, 2].set_title('Brier vs Robustness')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"pareto_front_{CONFIG['run_id']}.png"), dpi=150)
    plt.show()
    
    print("\n" + "="*80)
else:
    print("No results available. Run optimization first.")

## 7. Export Pareto Solutions

Export hyperparameters and metrics for all Pareto-optimal solutions

In [ ]:
if res is not None:
    print("="*80)
    print("PARETO SOLUTIONS EXPORT")
    print("="*80)
    
    pareto_data = []
    
    for i in range(len(res.X)):
        # Decode hyperparameters
        lr = 10 ** res.X[i, 0]  # log10 scale
        wd = 10 ** res.X[i, 1]  # log10 scale
        dropout = res.X[i, 2]
        aug_strength = res.X[i, 3]
        unfreeze_frac = res.X[i, 4]
        
        # Decode objectives (convert from minimization)
        pr_auc = -res.F[i, 0]
        auroc = -res.F[i, 1]
        brier = res.F[i, 2]
        robust_deg = res.F[i, 3]
        
        pareto_data.append({
            'solution_id': i,
            'learning_rate': lr,
            'weight_decay': wd,
            'dropout': dropout,
            'augmentation_strength': aug_strength,
            'unfreeze_fraction': unfreeze_frac,
            'pr_auc': pr_auc,
            'auroc': auroc,
            'brier': brier,
            'robustness_degradation': robust_deg
        })
    
    pareto_df = pd.DataFrame(pareto_data)
    pareto_csv = os.path.join(output_dir, f"pareto_solutions_{CONFIG['run_id']}.csv")
    pareto_df.to_csv(pareto_csv, index=False)
    
    print(f"\n✓ Pareto solutions exported to:")
    print(f"  {pareto_csv}")
    print(f"\n  Total solutions: {len(pareto_df)}")
    
    # Display summary
    print(f"\nMetrics Summary:")
    print(f"  PR-AUC: [{pareto_df['pr_auc'].min():.4f}, {pareto_df['pr_auc'].max():.4f}]")
    print(f"  AUROC: [{pareto_df['auroc'].min():.4f}, {pareto_df['auroc'].max():.4f}]")
    print(f"  Brier: [{pareto_df['brier'].min():.4f}, {pareto_df['brier'].max():.4f}]")
    print(f"  Robustness Degrad: [{pareto_df['robustness_degradation'].min():.4f}, {pareto_df['robustness_degradation'].max():.4f}]")
    
    print(f"\nHyperparameter Ranges:")
    print(f"  Learning Rate: [{pareto_df['learning_rate'].min():.6f}, {pareto_df['learning_rate'].max():.6f}]")
    print(f"  Weight Decay: [{pareto_df['weight_decay'].min():.6f}, {pareto_df['weight_decay'].max():.6f}]")
    print(f"  Dropout: [{pareto_df['dropout'].min():.4f}, {pareto_df['dropout'].max():.4f}]")
    print(f"  Aug Strength: [{pareto_df['augmentation_strength'].min():.4f}, {pareto_df['augmentation_strength'].max():.4f}]")
    print(f"  Unfreeze Fraction: [{pareto_df['unfreeze_fraction'].min():.4f}, {pareto_df['unfreeze_fraction'].max():.4f}]")
    print("="*80)
else:
    print("No results available. Run optimization first.")

## 8. Next Steps

After optimization completes:

1. **Review Pareto solutions** in the CSV file exported above
2. **Select representative solutions** based on your priorities:
   - High PR-AUC for malignancy detection
   - High AUROC for overall discrimination
   - Low Brier for calibration
   - Low robustness degradation for stability
3. **Retrain selected models** with the Pareto-optimal hyperparameters
4. **Evaluate on INbreast** for zero-shot transfer performance
5. **Compare with baseline** (single fixed hyperparameter configuration)
6. **Analyze trade-offs** between objectives

**Files Generated:**
- `results_{run_id}.pkl` - Full optimization results (pymoo Result object or final_results dict)
- `pareto_solutions_{run_id}.csv` - Pareto-optimal solutions with hyperparameters and metrics
- `pareto_front_{run_id}.png` - Visualization of Pareto front
- `checkpoint_{run_id}_gen{N}.pkl` - Checkpoints every 5 generations (surrogate mode only)

**Surrogate Mode Benefits:**
- 70% reduction in expensive CNN training evaluations
- Intelligent exploration using GP uncertainty quantification
- Balanced exploitation/exploration via uncertainty-weighted Pareto ranking
- Converges to high-quality Pareto front with significantly fewer evaluations